## **TMDB Movie Data Analysis using Spark and APIs**


In [16]:
import os
import time
import json
import sys
import requests
from dotenv import load_dotenv
from pyspark.sql import SparkSession, functions as F
import matplotlib.pyplot as plt
from pathlib import Path


### *Functions from scripts*

In [24]:
sys.path.append(os.path.abspath("..")) 
from src.tmdb_client import fetch_movie_with_credits
from src.bronze_to_spark import read_bronze_json


Let us start the Spark session.

In [18]:
spark = SparkSession.builder.appName("tmdb_lab").getOrCreate()
spark

## 1. **Fetching the movie data from the API**

In this project, we will follow the medallion workflow to ingest, transform and analyze the movie data for the wanted ids.

In [19]:
load_dotenv()

TMDB_API_KEY = os.getenv("TMDB_API_KEY")

print(f"The TMDB API Key is fetched")

The TMDB API Key is fetched


In [20]:
# TMDB base URLs and Endpoints
BASE_URL = "https://api.themoviedb.org/3"
MOVIE_ENDPOINT = f"{BASE_URL}/movie"
CREDITS_ENDPOINT = f"{BASE_URL}/movie"

# Movie IDs to fetch
movie_ids = [
    0, 299534, 19995, 140607, 299536, 597, 135397, 420818,
    24428, 168259, 99861, 284054, 12445, 181808, 330457,
    351286, 109445, 321612, 260513
]

len(movie_ids), movie_ids[:5]


(19, [0, 299534, 19995, 140607, 299536])

### **Bronze Layer:**

#### **Bronze Ingestion**
Let us ingest our raw data as JSON first

In [21]:
PROJECT_ROOT = Path(os.getcwd()).parent
out_dir = PROJECT_ROOT / "data" / "bronze" / "tmdb_raw_json"
out_dir.mkdir(parents=True, exist_ok=True)

movie_ids = [
    0, 299534, 19995, 140607, 299536, 597, 135397, 420818,
    24428, 168259, 99861, 284054, 12445, 181808, 330457,
    351286, 109445, 321612, 260513
]


In [22]:

for mid in movie_ids:
    out_path = out_dir / f"movie_id={mid}.json"

    if out_path.exists():
        print(f"Exists, skipping: {mid}")
        continue

    print(f"Fetching: {mid}")
    bundle = fetch_movie_with_credits(mid, TMDB_API_KEY)

    if bundle is None:
        print(f"Movie with Id: {mid} not found")
        continue

    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(bundle, f, ensure_ascii=False)

Fetching: 0
[HTTP 404] Failed: https://api.themoviedb.org/3/movie/0 | {'success': False, 'status_code': 6, 'status_message': 'Invalid id: The pre-requisite id is invalid or not found.'}
ID 0: Movie not found. Skipping
Movie with Id: 0 not found
Fetching: 299534
Fetching: 19995
Fetching: 140607
Fetching: 299536
Fetching: 597
Fetching: 135397
Fetching: 420818
Fetching: 24428
Fetching: 168259
Fetching: 99861
Fetching: 284054
Fetching: 12445
Fetching: 181808
Fetching: 330457
Fetching: 351286
Fetching: 109445
Fetching: 321612
Fetching: 260513


In [23]:
len(list(out_dir.glob("*.json"))) # Let us see the number of moies fetched

18

#### **Bronze to Spark**
Now we can read the Bronze JSON into Spark

In [26]:
bronze_path = str(out_dir)
bronze_df = read_bronze_json(spark, bronze_path)

In [27]:
bronze_df.printSchema()

bronze_df.select("movie.id", "movie.title", "credits.id","fetched_at").show(5, truncate=False)

print("Bronze count:", bronze_df.count())


root
 |-- credits: struct (nullable = true)
 |    |-- cast: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- adult: boolean (nullable = true)
 |    |    |    |-- cast_id: long (nullable = true)
 |    |    |    |-- character: string (nullable = true)
 |    |    |    |-- credit_id: string (nullable = true)
 |    |    |    |-- gender: long (nullable = true)
 |    |    |    |-- id: long (nullable = true)
 |    |    |    |-- known_for_department: string (nullable = true)
 |    |    |    |-- name: string (nullable = true)
 |    |    |    |-- order: long (nullable = true)
 |    |    |    |-- original_name: string (nullable = true)
 |    |    |    |-- popularity: double (nullable = true)
 |    |    |    |-- profile_path: string (nullable = true)
 |    |-- crew: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- adult: boolean (nullable = true)
 |    |    |    |-- credit_id: string (nullable = true)
 |

+------+-----------------------+------+----------+
|id    |title                  |id    |fetched_at|
+------+-----------------------+------+----------+
|19995 |Avatar                 |19995 |1769766563|
|299536|Avengers: Infinity War |299536|1769766568|
|24428 |The Avengers           |24428 |1769766574|
|99861 |Avengers: Age of Ultron|99861 |1769766580|
|299534|Avengers: Endgame      |299534|1769766560|
+------+-----------------------+------+----------+
only showing top 5 rows


26/01/30 11:59:09 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Bronze count: 18


All our movies are now stored into a spark data frame

## 2. **Silver Layer: Data Cleaning and Preprocessing**

Since we have our data in a dataframe now we can go ahead and clean it according to what we need from it
### **Data Preparation and Cleaning**